# YOLO 3회차 산업 데모 ② — 매장 동선 히트맵과 혼잡(대기열) 경보

매장 천장 카메라 영상에서 손님을 추적해 **어느 통로에 얼마나 머무는지**, **어디에 사람이 몰리는지**를 숫자와 지도로 만듭니다.
같은 방법이 계산대 줄(대기열) 관리, 전시장 관람 동선, 공장 통로 혼잡 관리에 그대로 쓰입니다.

| 단계 | 하는 일 | 결과물 |
|---|---|---|
| Step 1 | 손님 추적 + 발 위치 기록 | 관측 표 `obs` |
| Step 2 | 통로 구역별 방문·체류 시간 → 관심 손님 | `zone_visits.csv` |
| Step 3 | 혼잡(대기열) 경보 — N명 이상이 T초 넘게 머물면 알림 | `crowd_events.csv` |
| Step 4 | 동선 히트맵 — 직접 만들기 vs Ultralytics `solutions` 한 줄 | `retail_heatmap.png` |
| Step 5 | 매장 리포트와 관제 영상 | `retail_report.png`, `retail_monitor.mp4` |

**실습 영상**: Intel IoT DevKit 공개 샘플 `store-aisle-detection.mp4` (식기 매장 통로, 720×404, 60fps, 65초, CC BY 4.0 — 출처: github.com/intel-iot-devkit/sample-videos)
**실행 환경**: Google Colab, 런타임 유형 **T4 GPU**(Colab에서 빌려 쓰는 NVIDIA 그래픽카드)

## 0. 준비

In [ ]:
# 패키지 설치 — ultralytics(YOLO), lap(ByteTrack이 박스를 짝지을 때 사용), shapely(solutions의 구역 계산). -q: 설치 로그 줄임
!pip -q install ultralytics lap shapely

In [ ]:
import os, json, shutil, base64, subprocess, time
from pathlib import Path

# 영상·이미지 처리(cv2), 배열(numpy), 표(pandas), 그래프(matplotlib)
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display  # 셀 출력에 HTML(영상 플레이어 등) 띄우기
from ultralytics import YOLO

print('GPU 사용 가능:', torch.cuda.is_available())  # T4 런타임이면 True

try:
    from google.colab import drive  # Colab에만 있는 모듈 → 성공하면 Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True  # False면 Drive 대신 런타임 임시 폴더에 저장 (연결이 끊기면 사라짐)
if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_day3/retail')  # Drive 안 결과 폴더
else:
    SAVE_DIR = Path('outputs/retail')
SAVE_DIR.mkdir(parents=True, exist_ok=True)  # 중간 폴더까지 생성, 이미 있으면 그냥 통과
print('결과 저장 위치:', SAVE_DIR.resolve())

In [ ]:
# OpenCV(BGR) 이미지를 matplotlib로 보여 주기
def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')  # [..., ::-1]: 색 축을 뒤집어 BGR→RGB, 눈금 숨김
    if title:
        plt.title(title)
    plt.show()

# 여러 장을 격자로 보여 주기 (입력: BGR 이미지 목록, 제목 목록, 한 줄에 놓을 장 수)
def show_grid(imgs, titles=None, cols=3, size=4.5):
    rows = (len(imgs) + cols - 1) // cols  # 필요한 줄 수 (올림)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for k, im in enumerate(imgs):
        ax = axes.ravel()[k]
        ax.imshow(im[..., ::-1])
        if titles:
            ax.set_title(titles[k], fontsize=9)
    plt.tight_layout(); plt.show()

# mp4 영상을 노트북 안에서 재생 (입력: 영상 경로, 표시 폭 px)
# OpenCV가 만든 mp4v 영상은 브라우저가 못 틀어서 H.264로 다시 인코딩
def show_video(path, width=640):
    h264 = str(Path(path).with_name(Path(path).stem + '_h264.mp4'))
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(path),
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', h264], check=True)
    data = base64.b64encode(open(h264, 'rb').read()).decode()  # 영상을 글자(base64)로 바꿔 HTML에 직접 넣음
    display(HTML(f'<video width="{width}" controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

# 파일을 SAVE_DIR로 복사하고 복사된 경로를 돌려줌
def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():  # 원본과 대상이 같은 파일이면 복사 생략
        shutil.copy(path, dst)
    return dst

In [ ]:
# 실습 영상 내려받기 (Intel IoT DevKit 공개 샘플, CC BY 4.0)
VIDEO = 'store_aisle.mp4'  # 영문 파일 이름으로 저장
if not Path(VIDEO).exists():
    !wget -q https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/store-aisle-detection.mp4 -O {VIDEO}

cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS)  # 초당 프레임 수 (이 영상은 약 60)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.set(cv2.CAP_PROP_POS_FRAMES, N_FRAMES // 3)  # 사람이 보이는 1/3 지점으로 이동
ok, SAMPLE = cap.read()
cap.release()
print(f'{VIDEO}: {W}x{H}, {FPS:.0f}fps, {N_FRAMES}프레임, {N_FRAMES / FPS:.1f}초')
show_bgr(SAMPLE, 'sample frame')

---
## Step 1. 손님 추적 + 발 위치 기록

`model.track()`은 탐지에 **추적(tracking — 프레임이 바뀌어도 같은 사람에게 같은 번호를 붙이는 기술)** 을 더합니다.
이 영상은 60fps라 모든 프레임을 볼 필요가 없습니다. `VID_STRIDE = 3`이면 3프레임마다 하나씩(초당 20장) 처리해 3배 빨라집니다.
바닥 위 구역을 판정할 것이므로 박스 중심이 아니라 **발 위치**(박스 아래 가운데 점)를 기록합니다.

In [ ]:
model = YOLO('yolo26n.pt')  # COCO 사전학습 모델 (사람 = 클래스 0번)
CONF = 0.35       # 신뢰도(conf — 모델이 매긴 0~1 확신 점수) 기준
VID_STRIDE = 3    # 몇 프레임마다 하나씩 처리할지
DT = VID_STRIDE / FPS  # 처리하는 프레임 사이의 시간(초)

rows = []
t0 = time.time()
# stream=True: 결과를 한꺼번에 모으지 않고 프레임마다 하나씩 받음 (메모리 절약)
# persist=True: 이전 프레임의 추적 정보를 이어서 씀
for k, r in enumerate(model.track(VIDEO, stream=True, persist=True, classes=[0], conf=CONF,
                                  vid_stride=VID_STRIDE, tracker='bytetrack.yaml', verbose=False)):
    if r.boxes.id is None:  # 이 프레임에 추적 번호가 붙은 사람이 없음
        continue
    for (x1, y1, x2, y2), tid, cf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(),
                                         r.boxes.conf.cpu().numpy()):
        rows.append({'frame': k * VID_STRIDE, 'time_s': round(k * DT, 2), 'track_id': tid,
                     'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                     'foot_x': (x1 + x2) / 2, 'foot_y': y2, 'conf': cf})  # 발 위치 = 박스 아래 가운데
obs = pd.DataFrame(rows)
print(f'처리 {time.time() - t0:.0f}초 | 관측 {len(obs)}줄 | 서로 다른 track_id {obs.track_id.nunique()}개')
obs.head()

In [ ]:
# 시간별 인원 그래프: 같은 시각(time_s)의 관측 줄 수 = 화면 속 인원
people = obs.groupby('time_s').size().reindex(np.round(np.arange(0, obs.time_s.max() + DT, DT), 2), fill_value=0)
plt.figure(figsize=(10, 2.8)); plt.plot(people.index, people.values)
plt.xlabel('time (s)'); plt.ylabel('people'); plt.title('people in view'); plt.grid(alpha=.3); plt.show()

**확인해 보기** — track_id 개수가 실제 손님 수보다 많다면 가려짐 때문에 같은 사람의 번호가 바뀐 것(ID switch)입니다. 진열대에 몸이 가리는 매장 영상에서는 흔합니다. 그래서 아래 단계에서는 아주 짧게 나타난 번호를 빼고 셉니다.

---
## Step 2. 통로 구역별 방문·체류 시간

화면을 **왼쪽 통로 / 오른쪽 통로** 두 구역으로 나눕니다(가운데 진열 테이블을 사이에 둔 바닥). 꼭짓점 좌표는 픽셀 단위이며, 다른 영상에 쓸 때는 첫 화면을 보고 바꾸면 됩니다.
- **방문**: 한 track_id가 그 구역 안에서 `MIN_VISIT_S`초 이상 관측됨
- **관심 손님**: 한 구역에 `INTEREST_S`초 이상 머문 방문 — 진열 상품을 보고 있었을 가능성이 큼

In [ ]:
# ✏️ 구역 꼭짓점 (x, y) 픽셀 — 첫 화면을 보며 조정
ZONES = {
    'left_aisle':  np.array([[40, 404], [235, 404], [225, 130], [165, 130]], np.int32),
    'right_aisle': np.array([[420, 404], [640, 404], [500, 100], [440, 100]], np.int32),
}
MIN_VISIT_S = 1.0   # 이보다 짧은 관측은 지나친 것(또는 잘못된 번호)으로 보고 뺌
INTEREST_S = 3.0    # 이 시간 이상 머물면 '관심 손님'

view = SAMPLE.copy()
for name, poly in ZONES.items():
    cv2.polylines(view, [poly], True, (0, 200, 255), 2)
    cv2.putText(view, name, tuple(poly[3] + [0, -8]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)
show_bgr(view, 'zones')

# 관측 한 줄마다 어느 구역에 있는지 표시 (없으면 '-')
def which_zone(x, y):
    for name, poly in ZONES.items():
        if cv2.pointPolygonTest(poly, (float(x), float(y)), False) >= 0:  # 안(+1)·경계(0)
            return name
    return '-'
obs['zone'] = [which_zone(x, y) for x, y in zip(obs.foot_x, obs.foot_y)]

# 방문 표: (구역, track_id)마다 처음·마지막 관측 시각과 관측 시간
visits = (obs[obs.zone != '-'].groupby(['zone', 'track_id'])
          .agg(first_s=('time_s', 'min'), last_s=('time_s', 'max'), n_obs=('time_s', 'size')).reset_index())
visits['dwell_s'] = (visits['n_obs'] * DT).round(2)  # 구역 안에서 실제로 관측된 시간
visits = visits[visits['dwell_s'] >= MIN_VISIT_S]
visits['interested'] = visits['dwell_s'] >= INTEREST_S
visits.to_csv('zone_visits.csv', index=False); save_out('zone_visits.csv')  # 열 이름은 영문

zone_stats = visits.groupby('zone').agg(visitors=('track_id', 'nunique'), interested=('interested', 'sum'),
                                        mean_dwell_s=('dwell_s', 'mean'), max_dwell_s=('dwell_s', 'max')).round(1)
zone_stats['interest_rate'] = (zone_stats['interested'] / zone_stats['visitors']).round(2)  # 방문 중 관심 비율
zone_stats

**생각해 보기** — `interest_rate`(방문한 손님 중 오래 머문 비율)가 높은 통로는 상품이 눈길을 끈다는 뜻일 수도, 길이 막혀 지나가기 어렵다는 뜻일 수도 있습니다. 숫자 하나로 결론 내리지 말고 Step 4 히트맵과 함께 봅니다.

---
## Step 3. 혼잡(대기열) 경보

계산대 줄 관리의 기본 규칙은 "**N명 이상이 T초 넘게** 줄 서 있으면 계산대를 하나 더 연다"입니다. 통로에서도 같은 규칙으로 혼잡을 알립니다.
순간적으로 사람이 겹쳐 지나가는 것까지 경보로 울리지 않도록, 조건이 `CROWD_S`초 **연속으로** 이어질 때만 경보를 확정합니다(2회차 위험구역 실습의 연속 프레임 조건과 같은 생각).

In [ ]:
CROWD_N = 3    # ✏️ 구역 안 인원이 이 수 이상이면 '혼잡 상태'
CROWD_S = 2.0  # ✏️ 혼잡 상태가 이 시간(초) 넘게 이어지면 경보

times = np.round(np.arange(0, obs.time_s.max() + DT, DT), 2)  # 처리한 모든 시각
count = (obs[obs.zone != '-'].groupby(['time_s', 'zone']).size().unstack(fill_value=0)
         .reindex(index=times, columns=list(ZONES), fill_value=0))  # 시각 × 구역 인원 표

events = []
for z in ZONES:
    crowded = (count[z] >= CROWD_N).values
    start, run = None, 0  # 혼잡이 시작된 시각, 혼잡이 이어진 프레임 수
    for i, c in enumerate(list(crowded) + [False]):  # 끝에 False를 붙여 마지막 구간도 닫음
        if c:
            run += 1
            if run == 1:
                start = times[i]
        else:
            if run * DT >= CROWD_S:  # 연속 시간이 기준 이상이면 경보 확정
                events.append({'zone': z, 'start_s': float(start), 'end_s': float(times[i - 1]),
                               'duration_s': round(run * DT, 2), 'max_people': int(count[z].iloc[i - run:i].max())})
            run = 0
crowd = pd.DataFrame(events, columns=['zone', 'start_s', 'end_s', 'duration_s', 'max_people'])
crowd.to_csv('crowd_events.csv', index=False); save_out('crowd_events.csv')

fig, ax = plt.subplots(figsize=(10, 3))
for z in ZONES:
    ax.plot(times, count[z], label=z)
ax.axhline(CROWD_N, ls='--', c='gray', label=f'CROWD_N={CROWD_N}')
for _, e in crowd.iterrows():  # 경보 구간을 빨간 띠로 표시
    ax.axvspan(e.start_s, e.end_s, color='red', alpha=.15)
ax.set_xlabel('time (s)'); ax.set_ylabel('people in zone'); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f'crowd alerts: {len(crowd)}'); plt.show()
crowd

경보가 하나도 없으면 `CROWD_N`을 2로 낮춰 보고, 너무 많으면 `CROWD_S`를 늘려 보세요. 현장에서는 "경보가 하루 몇 번 울려야 직원이 무시하지 않는가"를 기준으로 이 두 값을 정합니다.

---
## Step 4. 동선 히트맵 — 직접 만들기 vs `solutions` 한 줄

**① 직접 만들기**: 발 위치를 빈 판에 점으로 찍고 흐리게 번지게 하면(가우시안 블러 — 각 점을 종 모양으로 주변에 퍼뜨리는 흐림 처리) 사람이 자주 지나간 곳이 진하게 남습니다.

**② Ultralytics `solutions`**: Ultralytics는 히트맵·구역 카운팅·대기열 관리·속도 추정 같은 현장 기능을 `solutions` 모듈로 미리 만들어 두었습니다. 몇 줄이면 바로 쓸 수 있지만, 규칙(관심 손님, 혼잡 경보)을 우리 문제에 맞게 바꾸려면 ①처럼 원리를 알아야 합니다.

In [ ]:
# ① 직접 만든 히트맵: 발 위치를 누적 → 블러 → 색 입혀 원본과 겹치기
acc = np.zeros((H, W), np.float32)
for x, y in zip(obs.foot_x.astype(int).clip(0, W - 1), obs.foot_y.astype(int).clip(0, H - 1)):
    acc[y, x] += 1
acc = cv2.GaussianBlur(acc, (0, 0), sigmaX=12)  # sigmaX가 클수록 넓게 번짐
norm = (255 * acc / acc.max()).astype(np.uint8)  # 0~255로 맞춤
color = cv2.applyColorMap(norm, cv2.COLORMAP_JET)  # 낮음 파랑 → 높음 빨강
mine = np.where(norm[..., None] > 10, cv2.addWeighted(SAMPLE, 0.5, color, 0.5, 0), SAMPLE)  # 거의 0인 곳은 원본 유지
for name, poly in ZONES.items():
    cv2.polylines(mine, [poly], True, (255, 255, 255), 1)
cv2.imwrite('retail_heatmap.png', mine); save_out('retail_heatmap.png')
show_bgr(mine, 'my heatmap (foot positions)')

In [ ]:
# ② Ultralytics solutions 히트맵: 프레임을 넣을 때마다 누적된 히트맵 그림(plot_im)을 돌려줌
import logging
from ultralytics import solutions
from ultralytics.utils import LOGGER
LOGGER.setLevel(logging.ERROR)  # 사람이 없는 프레임마다 나오는 'No tracks found' 경고 숨기기
heatmap = solutions.Heatmap(model='yolo26n.pt', classes=[0], colormap=cv2.COLORMAP_JET, show=False, verbose=False)
cap = cv2.VideoCapture(VIDEO)
k, last = 0, None
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if k % (VID_STRIDE * 2) == 0:  # 시간 절약: ①보다 2배 성기게 처리
        last = heatmap(frame).plot_im
    k += 1
cap.release()
LOGGER.setLevel(logging.INFO)  # 경고 표시 되돌리기
show_bgr(last, 'ultralytics solutions.Heatmap (last frame)')

두 그림의 차이를 보세요. `solutions.Heatmap`은 **사람 박스 영역**을 누적하므로 몸통 높이의 진열대 위까지 칠해지고, 직접 만든 것은 **발 위치**만 누적해 바닥 동선이 더 또렷합니다. 무엇을 누적할지는 질문(바닥 동선인가, 진열대 앞 노출인가)에 따라 정합니다.

---
## Step 5. 매장 리포트와 관제 영상

In [ ]:
# 리포트 그림: 왼쪽 = 구역별 방문·관심 손님, 오른쪽 = 히트맵
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1, 1.4]})
zs = zone_stats.reindex(list(ZONES)).fillna(0)
x = np.arange(len(zs))
axes[0].bar(x - 0.2, zs['visitors'], 0.4, label='visitors')
axes[0].bar(x + 0.2, zs['interested'], 0.4, label=f'dwell >= {INTEREST_S:.0f}s')
axes[0].set_xticks(x); axes[0].set_xticklabels(zs.index); axes[0].legend(); axes[0].grid(alpha=.3, axis='y')
axes[0].set_title(f'visits by zone | crowd alerts {len(crowd)}')
axes[1].imshow(mine[..., ::-1]); axes[1].axis('off'); axes[1].set_title('foot-position heatmap')
plt.tight_layout(); plt.savefig('retail_report.png', dpi=120); save_out('retail_report.png'); plt.show()

summary = {'video': VIDEO, 'seconds': round(N_FRAMES / FPS, 1), 'tracks': int(obs.track_id.nunique()),
           'zones': zone_stats.reset_index().to_dict('records'), 'crowd_alerts': len(crowd),
           'rules': {'MIN_VISIT_S': MIN_VISIT_S, 'INTEREST_S': INTEREST_S, 'CROWD_N': CROWD_N, 'CROWD_S': CROWD_S}}
Path('retail_summary.json').write_text(json.dumps(summary, indent=2, default=float)); save_out('retail_summary.json')
print(json.dumps(summary, indent=2, default=float))

In [ ]:
# 관제 영상: 처리한 프레임마다 구역·사람·구역별 인원·경보 표시
alert_at = lambda z, t: any((crowd.zone == z) & (crowd.start_s <= t) & (crowd.end_s >= t))  # 그 시각에 경보 중인가
by_time = {t: g for t, g in obs.groupby('time_s')}  # 시각 → 그 시각 관측 줄들
cap = cv2.VideoCapture(VIDEO)
out = cv2.VideoWriter('retail_monitor.mp4', cv2.VideoWriter_fourcc(*'mp4v'), FPS / VID_STRIDE, (W, H))
k = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if k % VID_STRIDE == 0:
        t = round(k / VID_STRIDE * DT, 2)
        g = by_time.get(t, obs.iloc[:0])
        for name, poly in ZONES.items():
            n = int((g.zone == name).sum())
            col = (0, 0, 255) if alert_at(name, t) else (0, 200, 255)
            cv2.polylines(frame, [poly], True, col, 2)
            cv2.putText(frame, f'{name}: {n}', tuple(poly[3] + [0, -8]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
        for _, o in g.iterrows():
            cv2.rectangle(frame, (int(o.x1), int(o.y1)), (int(o.x2), int(o.y2)), (0, 255, 0), 1)
            cv2.circle(frame, (int(o.foot_x), int(o.foot_y)), 4, (0, 255, 0), -1)
        cv2.putText(frame, f't={t:.1f}s', (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        out.write(frame)
    k += 1
cap.release(); out.release()
save_out('retail_monitor.mp4')
show_video('retail_monitor.mp4')

---
## 도전 과제

1. **계산대 줄** — 구역 하나를 '계산대 앞'이라 보고, 줄에 선 사람 수를 시간별로 기록해 "평균 대기 시간"(구역 체류 시간의 평균)을 구해 보세요.
2. **`solutions` 더 써 보기** — `solutions.RegionCounter`, `solutions.QueueManager`에 같은 구역 좌표를 넣어 결과를 우리가 만든 표와 비교해 보세요.
3. **워크숍 주제와 연결** — 매장 선반(T5) 주제라면 선반 빈칸 탐지와 이 노트북의 동선·체류를 묶어 "손님이 많이 머무는 통로의 빈 선반부터 채우기" 규칙을 만들 수 있습니다.

## 정리

- 추적 + 발 위치 + 구역 판정만으로 **방문·체류·혼잡**이라는 매장 지표가 나옵니다.
- 경보는 순간 값이 아니라 **연속 시간 조건**으로 확정해야 깜빡이지 않습니다.
- `solutions`는 빠른 출발점이고, 우리 문제에 맞는 규칙은 원리를 알아야 바꿀 수 있습니다.
- 사람이 찍힌 영상은 개인정보입니다. 실제 매장에서는 얼굴을 저장하지 않고 위치·시간 같은 집계 값만 남기는 설계가 필요합니다.

## 용어 정리

| 용어 | 뜻 |
|---|---|
| 추적(tracking)과 track_id | 프레임이 바뀌어도 같은 사람에게 같은 번호를 붙이는 기술과 그 번호입니다. |
| ByteTrack | Ultralytics가 제공하는 추적 방식입니다. 신뢰도가 낮은 박스도 매칭에 활용해 가려짐에 비교적 강합니다. |
| ID switch | 가려짐 때문에 같은 사람의 track_id가 중간에 바뀌는 현상입니다. |
| vid_stride | 영상에서 몇 프레임마다 하나씩 처리할지 정하는 값입니다. 3이면 3장 중 1장만 봅니다. |
| stream=True | 결과를 프레임마다 하나씩 내주는 옵션입니다. 긴 영상에서 메모리를 아낍니다. |
| 발 위치 | 박스 아래 가운데 점입니다. 사람이 서 있는 바닥 위치로 씁니다. |
| 구역(다각형) | 꼭짓점 좌표 목록으로 표현한 바닥 영역입니다. `cv2.pointPolygonTest`로 점이 안에 있는지 판정합니다. |
| 방문 / 체류 시간 | 한 사람이 한 구역에서 관측된 한 번의 기록 / 그 안에서 관측된 시간(초)입니다. |
| 관심 손님 | 한 구역에 `INTEREST_S`초 이상 머문 방문입니다. 진열 상품을 보고 있었을 가능성이 큽니다. |
| 혼잡(대기열) 경보 | 구역 인원이 `CROWD_N` 이상인 상태가 `CROWD_S`초 넘게 이어질 때 내는 알림입니다. |
| 히트맵 | 값이 많이 쌓인 곳을 뜨거운 색으로 칠한 밀도 지도입니다. |
| 가우시안 블러 | 각 점을 종 모양으로 주변에 번지게 하는 흐림 처리입니다. |
| Ultralytics solutions | 히트맵·구역 카운팅·대기열·속도 추정 등을 미리 만들어 둔 Ultralytics 모듈입니다. |
| conf (신뢰도) | 모델이 예측마다 매긴 0~1 확신 점수입니다. |
| fps | 초당 프레임 수입니다. 이 영상은 약 60fps입니다. |
| BGR | OpenCV의 색 순서(파랑, 초록, 빨강)입니다. 화면에 보일 때는 RGB로 뒤집습니다. |
| H.264 / ffmpeg | 브라우저가 재생할 수 있는 영상 압축 방식 / 영상 형식을 바꾸는 명령줄 도구입니다. |